In [3]:
import matplotlib.pyplot as plt
def lzw_encode(data: bytes, max_dict_size: int = 4096):
    if not data:
        return b''
    
    encoded = bytearray()
    n = len(data)
    pos = 0
    
    # Инициализируем словарь всеми возможными байтами
    dictionary = {bytes([i]): i for i in range(256)}
    next_index = 256
    
    current_phrase = b""
    
    while pos < n:
        # Добавляем следующий байт
        current_byte = bytes([data[pos]])
        new_phrase = current_phrase + current_byte
        
        if new_phrase in dictionary:
            if next_index < max_dict_size:
                # Фраза есть в словаре, продолжаем
                current_phrase = new_phrase
                pos += 1
            else:
                # Словарь полон, но фраза существует.
                # Выдаем код для текущей фразы и начинаем новую с текущего байта.
                if current_phrase:  # Защита от пустой строки
                    prefix_index = dictionary[current_phrase]
                    encoded.extend(prefix_index.to_bytes(2, 'big'))
                current_phrase = current_byte
                pos += 1
        else:
            # Выдаем индекс текущей фразы
            prefix_index = dictionary[current_phrase]
            encoded.extend(prefix_index.to_bytes(2, 'big'))
            
            # Добавляем новую фразу в словарь (если не превышен лимит)
            if new_phrase and next_index < max_dict_size:
                dictionary[new_phrase] = next_index
                next_index += 1
            
            # Сбрасываем текущую фразу
            current_phrase = current_byte
            pos += 1
    
    # Обработка остатка
    if current_phrase:
        prefix_index = dictionary[current_phrase]
        encoded.extend(prefix_index.to_bytes(2, 'big'))
    
    return bytes(encoded)


def lzw_decode(encoded: bytes, max_dict_size: int = 4096):
    if not encoded:
        return b''
    
    decoded = bytearray()
    n = len(encoded)
    i = 0
    
    # Инициализируем словарь всеми возможными байтами
    dictionary = {i: bytes([i]) for i in range(256)}
    next_index = 256
    
    # Читаем первый индекс
    if i + 2 > n:
        return b''
    
    prev_index = int.from_bytes(encoded[i:i+2], 'big')
    i += 2
    
    # Первая фраза
    prev_phrase = dictionary.get(prev_index, b"")
    decoded.extend(prev_phrase)
    
    while i + 2 <= n:
        # Читаем следующий индекс
        curr_index = int.from_bytes(encoded[i:i+2], 'big')
        i += 2
        
        # Получаем фразу
        if curr_index in dictionary:
            curr_phrase = dictionary[curr_index]
        elif curr_index == next_index and next_index < max_dict_size:
            # Специальный случай: фраза, которую только будем добавлять
            curr_phrase = prev_phrase + bytes([prev_phrase[0]])
        else:
            # Некорректный индекс
            break
        
        # Добавляем в декодированный вывод
        decoded.extend(curr_phrase)
        
        # Добавляем новую фразу в словарь
        if next_index < max_dict_size and prev_phrase:
            new_phrase = prev_phrase + bytes([curr_phrase[0]])
            dictionary[next_index] = new_phrase
            next_index += 1
        
        prev_phrase = curr_phrase
    
    return bytes(decoded)

import os
def get_code_lengths(node, length=0, lengths=None):
    # Обход дерева для получения длин кодов для каждого символа
    if lengths is None:
        lengths = {}
    
    if len(node) == 2:  # Лист (символ, частота)
        # Обработка случая одного символа
        lengths[node[0]] = max(1, length)
        return lengths

    if len(node) == 4:  # Узел (None, вес, лево, право)
        get_code_lengths(node[2], length + 1, lengths)
        get_code_lengths(node[3], length + 1, lengths)
    
    return lengths

def generate_canonical_codes(lengths):
    # Сортируем символы: сначала по длине кода, затем по значению символа
    sorted_symbols = sorted(lengths.keys(), key=lambda s: (lengths[s], s))
    
    codes = {}
    code = 0
    prev_len = 0
    
    for symbol in sorted_symbols:
        curr_len = lengths[symbol]
        # Сдвигаем код на разницу длин
        code <<= (curr_len - prev_len)
        
        # Формируем битовую строку
        codes[symbol] = format(code, f'0{curr_len}b')
        
        code += 1
        prev_len = curr_len
        
    return codes

def frequency_model(b):
    d = {}
    for i in b:
        d[i] = d.get(i, 0) + 1
    return d

def huffman_encode(data, model):
    # Сортируем очередь по частоте
    queue = sorted([[char, freq] for char, freq in model.items()], key=lambda x: x[1])
    
    # Строим дерево
    while len(queue) > 1:
        one = queue.pop(0)
        two = queue.pop(0)
        # Узел: [None, вес, левый_потомок, правый_потомок]
        parent = [None, one[1] + two[1], one, two]
        
        # Вставляем обратно, сохраняя сортировку по весу
        insert_position = 0
        for i, item in enumerate(queue):
            if parent[1] > item[1]:
                insert_position = i + 1
            else:
                break
        queue.insert(insert_position, parent)
    
    tree = queue[0] if queue else None
    
    # 1. Получаем длины кодов из дерева
    lengths = get_code_lengths(tree) if tree else {}
    
    # 2. Генерируем канонические коды по длинам
    codes = generate_canonical_codes(lengths)
    
    # 3. Кодируем данные
    encoded = ''.join(codes[char] for char in data)
    
    return codes, encoded, lengths

def huffman_decode(encoded, lengths):
    if not lengths or not encoded:
        return b""
    
    # Восстанавливаем канонические коды для декодирования
    codes = generate_canonical_codes(lengths)
    reverse_codes = {v: k for k, v in codes.items()}
    
    result = []
    buffer = ""
    
    # Оптимизация: максимальная длина кода для ограничения поиска
    max_len = max(lengths.values()) if lengths else 0
    
    for bit in encoded:
        buffer += bit
        # Проверяем, есть ли такой код в таблице
        if len(buffer) <= max_len and buffer in reverse_codes:
            result.append(reverse_codes[buffer])
            buffer = ""
    
    return bytes(result)

def bits_to_bytes(bit_string):
    if not bit_string:
        return b"", 0
        
    padding = (8 - len(bit_string) % 8) % 8
    bit_string += '0' * padding
    
    result = []
    for i in range(0, len(bit_string), 8):
        byte_chunk = bit_string[i:i+8]
        result.append(int(byte_chunk, 2))
    
    return bytes(result), padding

def bytes_to_bits(data, total_bits):
    bit_string = ''.join(f'{byte:08b}' for byte in data)
    return bit_string[:total_bits]

def write_huffman_file(filename, codes, encoded_bits, lengths):
    compressed_bytes, padding = bits_to_bytes(encoded_bits)
    unique_symbols = len(codes)
    
    with open(filename, 'wb') as f:
        # 1. Количество уникальных символов
        f.write(unique_symbols.to_bytes(4, 'big'))

        # 2. Таблица: Символ + Длина кода
        for byte_val in sorted(codes.keys()):
            f.write(bytes([byte_val]))
            f.write(bytes([lengths[byte_val]]))
        
        # 3. Служебная информация
        f.write(bytes([padding]))
        f.write(len(encoded_bits).to_bytes(4, 'big'))
        
        # 4. Сжатые данные
        f.write(compressed_bytes)

def read_huffman_file(filename):
    with open(filename, 'rb') as f:
        # 1. Количество уникальных символов
        num_symbols = int.from_bytes(f.read(4), 'big')
        
        # 2. Читаем длины кодов
        lengths = {}
        for _ in range(num_symbols):
            byte_val = f.read(1)[0]
            code_len = f.read(1)[0]
            lengths[byte_val] = code_len
        
        # 3. Служебная информация
        padding = f.read(1)[0]
        total_bits = int.from_bytes(f.read(4), 'big')
        
        # 4. Сжатые данные
        compressed_data = f.read()
        encoded_bits = bytes_to_bits(compressed_data, total_bits)
    
    return lengths, encoded_bits

def compress_file(input_path, output_path):
    with open(input_path, 'rb') as f:
        data = f.read()
    
    if not data:
        with open(output_path, 'wb') as f:
            f.write(bytes([0])) # 0 символов
        return

    model = frequency_model(data)
    codes, encoded_bits, lengths = huffman_encode(data, model)
    write_huffman_file(output_path, codes, encoded_bits, lengths)
    
    print(f"Оригинал: {len(data)} байт")

def decompress_file(input_path, output_path):
    lengths, encoded_bits = read_huffman_file(input_path)
    
    if not lengths:
        with open(output_path, 'wb') as f:
            pass
        return

    result = huffman_decode(encoded_bits, lengths)
    
    with open(output_path, 'wb') as f:
        f.write(result)
    
    print(f"Распаковано: {len(result)} байт")

def compress(input_path: str, output_path: str):
        with open(input_path, 'rb') as f:
            original_data = f.read()
        
        lzw_compressed = lzw_encode(original_data)

        model = frequency_model(lzw_compressed)

        codes, encoded_bits, lengths = huffman_encode(lzw_compressed, model)
        write_huffman_file(output_path, codes, encoded_bits, lengths)


def decompress(input_path: str, output_path: str):
    lengths, encoded_bits = read_huffman_file(input_path)
    huffman_decompressed = huffman_decode(encoded_bits, lengths)
    
    final_data = lzw_decode(huffman_decompressed)

    with open(output_path, 'wb') as f:
        f.write(final_data)

TEST_FILES = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/enwik7",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test2.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test3.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test4.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test5.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test6.raw",
]

TEST_FILES_COMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/enwik7_comp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test2_comp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test3_comp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test4_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test5_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test6_comp.raw",
]

TEST_FILES_DECOMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/enwik7_decomp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test2_decomp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test3_decomp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test4_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test5_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw+huffman/test6_decomp.raw",
]

from pathlib import Path

for file_path in range(len(TEST_FILES)):
    
    original_size = Path(TEST_FILES[file_path]).stat().st_size
    
    print(f"\nФайл: {file_path+1} ({original_size:,} байт)")

    # Сжатие
    compress(TEST_FILES[file_path], TEST_FILES_COMPRESS[file_path])
    compressed_size = Path(TEST_FILES_COMPRESS[file_path]).stat().st_size
    
    # Распаковка
    decompress(TEST_FILES_COMPRESS[file_path], TEST_FILES_DECOMPRESS[file_path])
    decompressed_size = Path(TEST_FILES_DECOMPRESS[file_path]).stat().st_size
    
    # Проверка целостности
    if decompressed_size == original_size:
        print("correct")
    else:
        print("fail")
    
    # Статистика
    ratio = (original_size / compressed_size) if compressed_size > 0 else 0
    print(f"Исходный размер: {original_size}\n")
    print(f"После сжатия: {compressed_size}\n")
    print(f"После распаковки: {decompressed_size}\n")
    print(f"Коэффициент сжатия: {ratio}\n")


Файл: 1 (10,000,000 байт)
correct
Исходный размер: 10000000

После сжатия: 8916365

После распаковки: 10000000

Коэффициент сжатия: 1.1215332705648546


Файл: 2 (371,816 байт)
correct
Исходный размер: 371816

После сжатия: 274892

После распаковки: 371816

Коэффициент сжатия: 1.3525893805567277


Файл: 3 (1,104,440 байт)
correct
Исходный размер: 1104440

После сжатия: 1127646

После распаковки: 1104440

Коэффициент сжатия: 0.9794208466132102


Файл: 4 (1,532,562 байт)
correct
Исходный размер: 1532562

После сжатия: 1767309

После распаковки: 1532562

Коэффициент сжатия: 0.8671726336481057


Файл: 5 (2,001,012 байт)
correct
Исходный размер: 2001012

После сжатия: 2337933

После распаковки: 2001012

Коэффициент сжатия: 0.855889368942566


Файл: 6 (1,806,348 байт)
correct
Исходный размер: 1806348

После сжатия: 2127915

После распаковки: 1806348

Коэффициент сжатия: 0.8488816517577065

